# Processing: Visualizing Sea Ice ERA5 Data

In this notebook, we'll visualize the pre-processed ERA5 sea ice atmospheric profiles. These profiles were generated in the previous notebook and can be used as input for pyRadtran simulations.

**Overview:**
- We load the pre-processed ERA5 data from `data/era5_sea_ice_to_libradtran.nc` (generated by `sea_ice_era5.ipynb`).
- The dataset contains mean temperature (`t`) and specific humidity (`q`) profiles binned by `season`, `valid_time` (decade), `sea_ice_cover`, and `pressure_level`.
- We visualize decadal trends in potential temperature and humidity, separated by season and sea ice fraction.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

ds_out = xr.open_dataset('data/era5_sea_ice_to_libradtran.nc', chunks='auto')

In [ ]:
ds_out

In [ ]:
ds_trend = ds_out - ds_out.mean(dim='valid_time') 
ds_out['theta'] = ds_out['t'] * (1000 / ds_out['pressure_level'])**0.286 # dry potential temperature

## Potential Temperature Trends

Compute dry potential temperature and fit a linear trend across decades. The contour plots show the warming rate (K/decade) as a function of sea ice cover and pressure level, faceted by season.

In [ ]:
fig = plt.figure(figsize=(12, 10))

da_to_plot = ds_out.polyfit(dim='valid_time', deg=1).isel(degree=0).theta_polyfit_coefficients * (1e9 * 3600 * 24 * 365 * 10)
da_to_plot.plot.contourf(y='pressure_level', ylim=(1000, 100), yscale='log', col='season', robust=False, levels=np.arange(-1, 1, .1), cmap='RdBu_r')
plt.show()

In [ ]:
da_to_plot.sel(season='DJF').plot.contourf(
    ylim=(1000, 500), yscale='log', y='pressure_level', add_labels=False, levels=20, xlim=(0.7, 1)
)

In [ ]:
fig = plt.figure(figsize=(12, 10))

da_to_plot = ds_out.polyfit(dim='valid_time', deg=1).isel(degree=0).q_polyfit_coefficients * (1e9 * 3600 * 24 * 365 * 10) * 1000
da_to_plot.plot.contourf(y='pressure_level', ylim=(1000, 100), yscale='log', col='season', robust=True, cmap='cividis', levels=30)
plt.show()
da_to_plot.mean('season').plot(ylim=(1000, 100), yscale='log', y='pressure_level')

In [ ]:
ds_out.t.isel(season=0).plot.contourf(
    levels=np.arange(245, 270, 2.5),    
    cmap='hot',
    col='valid_time', col_wrap=4, yscale='log', y='pressure_level', add_labels=False, ylim=(1000, 100))

plt.ylim(1000, 700)

In [ ]:
ds_trend.q.isel(season=0).plot(col='valid_time', col_wrap=4, yscale='log', y='pressure_level', add_labels=False, ylim=(1000, 100))
ds_trend.t.isel(season=0).plot(col='valid_time', col_wrap=4, yscale='log', y='pressure_level', add_labels=False, ylim=(1000, 100))
# set all yscales to logscale 